In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 237 (delta 105), reused 186 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 798.23 KiB | 3.97 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 158.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 248.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 243.7 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 46.9 MB/s eta 0:00:00


In [4]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo, TileType)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class Direction(Enum):
    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3

In [7]:
from mario_the_explorer.environment import tiles
class TryThingsRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()
        self._block_action_counts = {}

    def reset(self) -> None:
        self._blocks_seen = set()
        self._block_action_counts = {}

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        reward = 0.0
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    reward += 10.0
        tiles_around_mario = self._get_tiles_around_mario(observation)
        for tile_and_direction in tiles_around_mario:
            if tile_and_direction not in self._block_action_counts:
                self._block_action_counts[tile_and_direction] = 0
            self._block_action_counts[tile_and_direction] += 1
            reward += 1.0 / (self._block_action_counts[tile_and_direction]**2)
        return reward

    def _get_tiles_around_mario(self, observation: list[list[Tile]]) -> set[tuple[Direction, int]]:
        mario_coordinates = self._find_mario_coordinates(observation)
        blocks_around_mario = set()
        if not mario_coordinates:
            return blocks_around_mario
        for mario_row, mario_col in mario_coordinates:
            if mario_row > 0:
                block_above_mario = observation[mario_row - 1][mario_col]
                if block_above_mario["type"] != TileType.MARIO and block_above_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.UP.name, tile_absolute_id(block_above_mario)))
            if mario_row < len(observation) - 1:
                block_below_mario = observation[mario_row + 1][mario_col]
                if block_below_mario["type"] != TileType.MARIO and block_below_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.DOWN.name, tile_absolute_id(block_below_mario)))
            if mario_col > 0:
                block_left_of_mario = observation[mario_row][mario_col - 1]
                if block_left_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.LEFT.name, tile_absolute_id(block_left_of_mario)))
            if mario_col < len(observation[0]) - 1:
                block_right_of_mario = observation[mario_row][mario_col + 1]
                if block_right_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.RIGHT.name, tile_absolute_id(block_right_of_mario)))
        return blocks_around_mario


    def _find_mario_coordinates(self, observation: list[list[Tile]]) -> list[tuple[int, int]]:
        mario_coordinates = []
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO:
                    mario_coordinates.append((row_idx, col_idx))
        return mario_coordinates

    def get_directional_heatmaps(self, observation: list[list[Tile]]) -> np.ndarray:
        h, w = len(observation), len(observation[0])
        heatmaps = np.zeros((4, h, w), dtype=np.float32)
        directions = ["UP", "DOWN", "LEFT", "RIGHT"]
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO or tile["type"] == TileType.EMPTY:
                    heatmaps[:, row_idx, col_idx] = 1.0
                    continue
                t_id = tile_absolute_id(tile)
                for i, d_name in enumerate(directions):
                    count = self._block_action_counts.get((d_name, t_id), 0)
                    heatmaps[i, row_idx, col_idx] = min(count / 10.0, 1.0)
        return heatmaps

In [8]:
class MarioReshapeWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Get the original H, W (14, 16)
        h, w = self.observation_space.shape
        # Redefine the space to include a single channel (1, 14, 16)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(5, h, w),
            dtype=np.float32
        )

    def observation(self, obs):
        # Add the channel dimension
        tile_channel = np.expand_dims(obs, axis=0).astype(np.float32)
        reward_model = self.env.unwrapped.reward_model
        heatmap = reward_model.get_directional_heatmaps(self.env.unwrapped.observation)
        heatmap_channel = heatmap.astype(np.float32)
        return np.concatenate((tile_channel, heatmap_channel), axis=0)

In [9]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import torch.nn as nn

class CustomMarioCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 16, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            sample_tensor = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_tensor).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))

In [10]:
RUN_NAME = "cnn_action_rewards"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 1

In [11]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [12]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = TryThingsRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
env = MarioReshapeWrapper(env)

2026-05-05 03:30:36 [INFO] Session log for run cnn_action_rewards with level [INFO] initialized at: cnn_action_rewards_20260505_033036.log


In [13]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=10000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

Using cpu device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 10000`, after every 156 untruncated mini-batches, there will be a truncated mini-batch of size 16
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=10000 and n_envs=1)
  warnings.warn(
2026-05-05 03:30:43 [INFO] Priming policy to prefer 'RIGHT_RUN'
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-05-05 03:30:43 [INFO] Priming complete
2026-05-05 03:30:43 [INFO] Starting training...
2026-05-05 03:31:31 [INFO] Step 10000 - time/iterations: 1
2026-05-05 03:31:31 [INFO] Step 10000 - time/f

In [14]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-05 03:37:00 [INFO] Step: 1000
2026-05-05 03:37:07 [INFO] Step: 2000
2026-05-05 03:37:13 [INFO] Step: 3000
2026-05-05 03:37:20 [INFO] Step: 4000
2026-05-05 03:37:26 [INFO] Step: 5000
2026-05-05 03:37:32 [INFO] Step: 6000
2026-05-05 03:37:39 [INFO] Step: 7000
2026-05-05 03:37:45 [INFO] Step: 8000
2026-05-05 03:37:52 [INFO] Step: 9000
2026-05-05 03:37:58 [INFO] Step: 10000
2026-05-05 03:38:04 [INFO] Step: 11000
2026-05-05 03:38:11 [INFO] Step: 12000
2026-05-05 03:38:17 [INFO] Step: 13000
2026-05-05 03:38:24 [INFO] Step: 14000
2026-05-05 03:38:30 [INFO] Step: 15000
2026-05-05 03:38:36 [INFO] Step: 16000
2026-05-05 03:38:40 [INFO] Terminated: True
2026-05-05 03:38:40 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/mo

In [15]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=32000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=500000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

2026-05-05 03:39:57 [INFO] Priming policy to prefer 'RIGHT_RUN'
2026-05-05 03:39:57 [INFO] Priming complete
2026-05-05 03:39:57 [INFO] Starting training...


Using cpu device


2026-05-05 03:42:33 [INFO] Step 32000 - train/learning_rate: 0.0003
2026-05-05 03:42:33 [INFO] Step 32000 - train/entropy_loss: -2.3197263536939197
2026-05-05 03:42:33 [INFO] Step 32000 - train/policy_gradient_loss: -0.0042216101475949785
2026-05-05 03:42:33 [INFO] Step 32000 - train/value_loss: 4.6153585587717165
2026-05-05 03:42:33 [INFO] Step 32000 - train/approx_kl: 0.009279865771532059
2026-05-05 03:42:33 [INFO] Step 32000 - train/clip_fraction: 0.0887937898089172
2026-05-05 03:42:33 [INFO] Step 32000 - train/loss: 4.483847618103027
2026-05-05 03:42:33 [INFO] Step 32000 - train/explained_variance: 0.5187000632286072
2026-05-05 03:42:33 [INFO] Step 32000 - train/n_updates: 60
2026-05-05 03:42:33 [INFO] Step 32000 - train/clip_range: 0.2
2026-05-05 03:42:33 [INFO] Step 32000 - time/iterations: 1
2026-05-05 03:42:33 [INFO] Step 32000 - time/fps: 205
2026-05-05 03:42:33 [INFO] Step 32000 - time/time_elapsed: 155
2026-05-05 03:42:33 [INFO] Step 32000 - time/total_timesteps: 32000
2026-

In [16]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trained-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-05 04:32:52 [INFO] Terminated: True
2026-05-05 04:32:52 [INFO] Truncated: False


In [17]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=20000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=5000000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 20000`, after every 312 untruncated mini-batches, there will be a truncated mini-batch of size 32
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=20000 and n_envs=1)
  warnings.warn(
2026-05-05 04:32:53 [INFO] Priming policy to prefer 'RIGHT_RUN'
2026-05-05 04:32:53 [INFO] Priming complete
2026-05-05 04:32:53 [INFO] Starting training...


Using cpu device


2026-05-05 04:34:31 [INFO] Step 20000 - train/learning_rate: 0.0003
2026-05-05 04:34:31 [INFO] Step 20000 - train/entropy_loss: -2.105401034736633
2026-05-05 04:34:31 [INFO] Step 20000 - train/policy_gradient_loss: -0.0020656941629480573
2026-05-05 04:34:31 [INFO] Step 20000 - train/value_loss: 27.18909194688797
2026-05-05 04:34:31 [INFO] Step 20000 - train/approx_kl: 0.010141225531697273
2026-05-05 04:34:31 [INFO] Step 20000 - train/clip_fraction: 0.100246875
2026-05-05 04:34:31 [INFO] Step 20000 - train/loss: 8.770848274230957
2026-05-05 04:34:31 [INFO] Step 20000 - train/explained_variance: 0.8130049109458923
2026-05-05 04:34:31 [INFO] Step 20000 - train/n_updates: 160
2026-05-05 04:34:31 [INFO] Step 20000 - train/clip_range: 0.2
2026-05-05 04:34:31 [INFO] Step 20000 - time/iterations: 1
2026-05-05 04:34:31 [INFO] Step 20000 - time/fps: 205
2026-05-05 04:34:31 [INFO] Step 20000 - time/time_elapsed: 97
2026-05-05 04:34:31 [INFO] Step 20000 - time/total_timesteps: 20000
2026-05-05 04:

In [18]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"full-train-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

2026-05-05 13:08:33 [INFO] Terminated: True
2026-05-05 13:08:33 [INFO] Truncated: False
